In [4]:
from pathlib import Path
from ultralytics import YOLO
import json

# === CONFIG CHUNG ===

# Thư mục project hiện tại (root repo)
ROOT_DIR = Path.cwd().parent    # vì notebook nằm trong Jupyter/YOLO_33

print("ROOT_DIR:", ROOT_DIR)

# Thư mục ảnh 33 món ăn (mỗi món 1 folder)
DATA_DIR = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/Images_cls")
print("DATA_DIR tồn tại?", DATA_DIR.exists())

# Thư mục YOLO auto-split sẽ tạo (train/val)
SPLIT_DIR = DATA_DIR.parent / "Images_split"
print("SPLIT_DIR:", SPLIT_DIR)

# Thư mục lưu các run YOLO-CLS 33
RUNS_ROOT  = ROOT_DIR / "YOLO_33/runs_yolo_cls33"
RUN_NAME   = "MTL_FOOD33_CLS_01"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print("RUNS_ROOT:", RUNS_ROOT)


ROOT_DIR: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter
DATA_DIR tồn tại? True
SPLIT_DIR: /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/Images_split
RUNS_ROOT: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/YOLO_33/runs_yolo_cls33


In [5]:
# Liệt kê 1 vài class để chắc chắn dataset ok
classes = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print("Tổng số lớp:", len(classes))
print(classes[:10], "...")

# Lưu luôn tên class để sau này app dùng
runs_meta_dir = ROOT_DIR / "YOLO_33/runs_meta"
runs_meta_dir.mkdir(exist_ok=True)

cls_json_path = runs_meta_dir / "yolo_cls33_class_names.json"
with open(cls_json_path, "w", encoding="utf-8") as f:
    json.dump(classes, f, ensure_ascii=False, indent=2)

print("✅ Đã lưu class_names vào:", cls_json_path)


Tổng số lớp: 2
['train', 'val'] ...
✅ Đã lưu class_names vào: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/YOLO_33/runs_meta/yolo_cls33_class_names.json


In [ ]:
MODEL = "yolo11s-cls.pt"   # hoặc yolo11m-cls.pt nếu GPU khỏe

print("MODEL:", MODEL)

model = YOLO(MODEL)

# Nếu SPLIT_DIR chưa tồn tại, YOLO sẽ tự chia 80/20.
# Nếu đã có, nó xài lại, không chia nữa.
print("Bắt đầu train YOLO-CLS 33 lớp...")

results = model.train(
    data=str(DATA_DIR),    # 👉 CHỈ ĐIỂM TỚI FOLDER GỐC, KHÔNG PHẢI YAML
    epochs=17,
    imgsz=224,
    batch=32,
    device=0,              # 0 = GPU
    project=str(RUNS_ROOT),
    name=RUN_NAME,
)

print("✅ Train xong.")


In [ ]:
BEST_WEIGHTS = RUNS_ROOT / RUN_NAME / "weights" / "best.pt"
print("BEST_WEIGHTS:", BEST_WEIGHTS, "tồn tại?", BEST_WEIGHTS.exists())

cls_model = YOLO(str(BEST_WEIGHTS))
print("✅ Loaded best YOLO-CLS:", BEST_WEIGHTS)
